# EDA: Diabetes 130-US Hospital Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.insert(0, '..')
from src.preprocessing import load_raw, clean, engineer_features, make_target, make_treatment, get_model_arrays

In [ ]:
df_raw = load_raw()
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
print("Missing values (top 15):")
df_raw.isnull().sum().sort_values(ascending=False).head(15)

In [ ]:
df = clean(df_raw)
print(f"After cleaning: {df.shape}")
print(f"Treatment=1 (home health): {(df.discharge_disposition_id==6).sum()}")
print(f"Treatment=0 (routine home): {(df.discharge_disposition_id==1).sum()}")

In [ ]:
target = make_target(df)
print(f"Readmission <30 days: {target.sum()} ({target.mean():.1%})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
# Readmission rate by age
df_eng = engineer_features(df.copy())
df_eng['readmitted_30'] = make_target(df)
df_eng.groupby('age')['readmitted_30'].mean().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Readmission Rate by Age Group'); axes[0].set_ylabel('30-day readmission rate'); axes[0].tick_params(axis='x', rotation=45)
# By race
df_eng.groupby('race')['readmitted_30'].mean().sort_values().plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Readmission Rate by Race')
# Prior visits distribution
df_eng['prior_visits'] = df_eng['number_outpatient'] + df_eng['number_emergency'] + df_eng['number_inpatient']
df_eng['prior_visits'].clip(0,8).value_counts().sort_index().plot(kind='bar', ax=axes[2], color='coral')
axes[2].set_title('Distribution of Prior Visits'); axes[2].set_xlabel('Prior visits')
plt.tight_layout(); plt.savefig('../data/processed/eda_plots.png', dpi=100, bbox_inches='tight'); plt.show()

## Feature Correlations with Target

In [ ]:
X, y, treatment = get_model_arrays(df)
correlations = X.corrwith(y).sort_values(ascending=False)
print("Feature correlations with 30-day readmission:")
print(correlations)

## Treatment Group Balance

In [ ]:
print(f"Treatment group size: {treatment.sum()}")
print(f"Control group size: {(treatment==0).sum()}")
X_treated = X[treatment==1]; X_control = X[treatment==0]
balance = pd.DataFrame({'treated_mean': X_treated.mean(), 'control_mean': X_control.mean()})
balance['std_diff'] = (balance['treated_mean'] - balance['control_mean']) / X.std()
print("\nStandardized mean differences (>0.1 = imbalance):")
print(balance.sort_values('std_diff', key=abs, ascending=False))